In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")

volume = f"{catalog}.{schema}.{volume_name}"
volume_path = f"{catalog}/{schema}/{volume_name}"

assert catalog, f"catalog:{catalog} is empty"
assert schema, f"schema:{schema} is empty"
assert volume_name, f"volume:{volume_path} is empty"

In [0]:
import datetime

run_date_str = dbutils.widgets.get("run_date")

if run_date_str:
    today = datetime.datetime.strptime(run_date_str, "%Y%m%d").date()
else:
    today = datetime.date.today()

# Monday of current week
current_monday = today - datetime.timedelta(days=today.weekday())
week_start = current_monday.strftime("%Y%m%d")

eq_summary_table = f"{catalog}.{schema}.earthquake_weekly_summary_brnz_tbl"
eq_details_table = f"{catalog}.{schema}.earthquake_details_brnz_tbl"

summary_df = (
    spark.table(eq_summary_table)
    .select("earthquake_id", "detail")
    .distinct()
)

# Check if details table exists
if spark.catalog.tableExists(eq_details_table):

    details_df = (
        spark.table(eq_details_table)
        .select("earthquake_id")
        .distinct()
    )

    new_ids_df = (
        summary_df.join(
            details_df,
            on="earthquake_id",
            how="left_anti"
        )
        .select("earthquake_id", "detail")
        .withColumnRenamed("detail", "details_url")
        .distinct()
    )

else:
    new_ids_df = (
        summary_df.select("earthquake_id", "detail")
        .withColumnRenamed("detail", "details_url")
        .distinct()
    )


In [0]:
%skip
import requests
import json
from pyspark.sql.types import StructType, StructField, StringType

urls_df = new_ids_df.select("details_url").repartition(200)

def fetch_partition(iterator):
    session = requests.Session()

    for row in iterator:
        url = row.details_url
        try:
            response = session.get(url, timeout=10)
            response.raise_for_status()

            yield {
                "details_url": url,
                "response_json": json.dumps(response.json())
            }

        except Exception:
            yield {
                "details_url": url,
                "response_json": None
            }

results_rdd = urls_df.rdd.mapPartitions(fetch_partition)

schema = StructType([
    StructField("details_url", StringType(), True),
    StructField("response_json", StringType(), True)
])

details_df = spark.createDataFrame(results_rdd, schema)

details_df.write.mode("append").json(
    f"/Volumes/{catalog}/{schema}/{volume_name}/details/week_start={week_start}"
)

In [0]:
import pandas as pd
import json
import requests
import time
import random

from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import struct, col

urls_df = new_ids_df.select("details_url").repartition(50)

details_schema = StructType([
    StructField("details_url", StringType(), True),
    StructField("response_json", StringType(), True)
])

MAX_WORKERS = 20
MAX_RETRIES = 3

def fetch_url(session, url):
    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(url, timeout=10)
            r.raise_for_status()
            return {
                "details_url": url,
                "response_json": json.dumps(r.json())
            }
        except Exception:
            if attempt == MAX_RETRIES - 1:
                return {
                    "details_url": url,
                    "response_json": None
                }
            time.sleep(1 + random.random())

def process_partition(iterator):
    session = requests.Session()
    for pdf in iterator:
        urls = pdf["details_url"].tolist()
        results = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(fetch_url, session, url)
                for url in urls
            ]
            for future in as_completed(futures):
                results.append(future.result())
        yield pd.DataFrame(results)

details_df = urls_df.mapInPandas(process_partition, details_schema)

details_write_path = f"/Volumes/{catalog}/{schema}/{volume_name}/details/week_start={week_start}"
details_df.write.mode("overwrite").parquet(details_write_path)

